In [9]:
import dolfinx
from mpi4py import MPI
from ufl import (TrialFunction, TestFunction, dot, grad, conditional,
                 FacetNormal, Measure, avg, jump, lhs, rhs)
import numpy as np
from dolfinx.fem.petsc import LinearProblem

# 1. Criação da malha estruturada 2D (Quadrados de 0 a 1)
mesh = dolfinx.mesh.create_unit_square(MPI.COMM_WORLD, 32, 32)

# 2. Definição do Espaço Funcional DG0 (Polinômios constantes por elemento)
V_space = dolfinx.fem.functionspace(mesh, ("DG", 0))

T = TrialFunction(V_space)
theta = TestFunction(V_space)

# 3. Constantes Físicas do Problema
D = dolfinx.fem.Constant(mesh, 0.01)       # Coeficiente de Difusão
V_vel = dolfinx.fem.Constant(mesh, (1.0, 0.0))  # Velocidade (Advecção pura da esquerda p/ direita)
f_T = dolfinx.fem.Constant(mesh, 0.0)       # Sem termo de fonte volumétrico para testar advecção

n = FacetNormal(mesh)  # Vetor normal à face

# 4. Configuração das medidas (Integrais de volume, faces internas e contorno)
dx = Measure("dx", domain=mesh)
dS = Measure("dS", domain=mesh)
ds = Measure("ds", domain=mesh)
# ==========================================================
# 5. FORMULAÇÃO VARIACIONAL (MÉTODO DG) - CORRIGIDA
# ==========================================================
vn = dot(V_vel, n)

# --- TERMO ADVECTIVO (FLUXO UPWIND INTERIOR) ---
# Usar vn (sem restrição) dentro do jump()
# Usar a restrição n('+') apenas para calcular o valor absoluto do fluxo
vn_plus = dot(V_vel, n('+'))

F_advection_interfaces = (
    avg(T) * jump(theta * vn) + 0.5 * abs(vn_plus) * jump(T) * jump(theta)
) * dS

# --- CONDIÇÃO DE CONTORNO ADVECTIVA ---
is_inlet = x[0] < 1e-8
inlet_factor = conditional(is_inlet, 1.0, 0.0)

# Na entrada (inlet): T_inlet | Na saída (outlet): T (fluxo de saída natural)
F_advection_boundaries = (
    (inlet_factor * T_inlet + (1.0 - inlet_factor) * T) * theta * vn
) * ds

# --- TERMO DIFUSIVO (INTERIOR PENALTY) ---
# h_avg calcula a dimensão característica média da interface para estabilização
from ufl import CellVolume, FacetArea
h = CellVolume(mesh) / FacetArea(mesh)
sigma = 10.0  # Parâmetro de penalização estrita para acoplamento DG

F_diffusion_interfaces = (
    - dot(avg(D * grad(T)), n('+')) * jump(theta) * dS
    - dot(avg(D * grad(theta)), n('+')) * jump(T) * dS
    + (sigma / avg(h)) * jump(T) * jump(theta) * dS
)

# Condição de Dirichlet homogênea fraca para difusão na borda esquerda
# Correção: aplique o conditional dentro da integral
inlet_factor = conditional(is_inlet, 1.0, 0.0)

F_diffusion_boundaries = (
    - dot(D * grad(T), n) * theta * inlet_factor
    - dot(D * grad(theta), n) * (T - T_inlet) * inlet_factor
    + (sigma / h) * (T - T_inlet) * theta * inlet_factor
) * ds
# ==========================================================
# 6. MONTAGEM DO SISTEMA E RESOLUÇÃO
# ==========================================================
Residual = - f_T * theta * dx + F_advection_interfaces + F_advection_boundaries + F_diffusion_interfaces + F_diffusion_boundaries

a_form = lhs(Residual)
L_form = rhs(Residual)

# 1. Criar a função que vai armazenar a solução resolvida
T_sol = dolfinx.fem.Function(V_space)
T_sol.name = "Temperatura"

# 2. Configurar o problema com os parâmetros exigidos pelas versões recentes
problem = LinearProblem(
    a_form, 
    L_form,
    bcs=[],               # Lista de contornos forte vazia (usamos condições fracas em DG)
    u=T_sol,              # Função de destino para a solução
    petsc_options={"ksp_type": "preonly", "pc_type": "lu"},
    petsc_options_prefix="dg_solver_"  # Prefixo exigido pelo erro
)

# 3. Executar o solver (ele atualiza a variável T_sol automaticamente)
problem.solve()

# ==========================================================
# 7. EXPORTAÇÃO DOS RESULTADOS (PARA VISUALIZAÇÃO)
# ==========================================================
from dolfinx.io import VTXWriter
with VTXWriter(mesh.comm, "resultado_dg0.bp", [T_sol], engine="BP4") as vtx:
    vtx.write(0.0)

print("Simulação concluída com sucesso! Arquivo 'resultado_dg0.bp' gerado.")

Simulação concluída com sucesso! Arquivo 'resultado_dg0.bp' gerado.


# Formulação Matemática e Discretização 
## Método de Galerkin Descontínuo (DG0)

### O Problema de Valor de Contorno
O código resolve a equação de advecção-difusão em estado estacionário no domínio bidimensional $\Omega = (0,1) \times (0,1)$. O problema forte é dado por:

$$
\nabla \cdot (\mathbf{v} T) - \nabla \cdot (D \nabla T) = f \quad \text{em } \Omega
$$

onde $\mathbf{v}$ é o campo de velocidades (advecção pura no eixo $x$, $\mathbf{v} = (1, 0)$), $D$ é o coeficiente de difusão constante, $T$ é a temperatura (ou concentração) e $f$ é o termo fonte. 

As fronteiras são particionadas em entrada ($\Gamma_{in}$, onde $\mathbf{v} \cdot \mathbf{n} < 0$) e saída ($\Gamma_{out}$, onde $\mathbf{v} \cdot \mathbf{n} \ge 0$). O problema está sujeito à seguinte condição de contorno de Dirichlet imposta de forma fraca na entrada:

$$
T = T_{inlet} \quad \text{em } \Gamma_{in}
$$

### Espaços de Funções
Seja $\mathcal{T}_h$ uma partição (malha) do domínio $\Omega$ em elementos $K$. Denotamos por $\mathcal{E}_i$ o conjunto de faces internas e por $\mathcal{E}_b$ o conjunto de faces de contorno.
O espaço funcional contínuo (broken Sobolev space) adequado para a formulação é:

$$
V = \{ v \in L^2(\Omega) : v|_K \in H^1(K) \quad \forall K \in \mathcal{T}_h \}
$$

Para a discretização espacial via Elementos Finitos, utilizamos o subespaço de dimensão finita correspondente a polinômios de grau $p$:

$$
V_h = \{ v \in L^2(\Omega) : v|_K \in \mathcal{P}_p(K) \quad \forall K \in \mathcal{T}_h \}
$$

No caso do código fornecido, adota-se $p=0$ (DG0), resultando em funções constantes por partes.

### Formulação Variacional Geral (DG)
Multiplicando a equação forte por uma função de teste $\theta \in V_h$ e integrando por partes elemento a elemento, obtemos a formulação fraca. Para os fluxos nas interfaces do método DG, adota-se o esquema *Upwind* para a advecção e o *Symmetric Interior Penalty Galerkin* (SIPG) para a difusão. O problema variacional consiste em encontrar $T \in V_h$ tal que $a(T, \theta) = L(\theta)$ para todo $\theta \in V_h$, onde as formas são separadas nos termos implementados no código:

#### Termos de Advecção
$$
\begin{align}
    \text{Interfaces: } F_{adv}^{int} &= \int_{\mathcal{E}_i} \left( \{T\} \llbracket \theta \mathbf{v} \cdot \mathbf{n} \rrbracket + \frac{1}{2} \|\mathbf{v} \cdot \mathbf{n}| \llbracket T \rrbracket \llbracket \theta \rrbracket \right) dS \\
    \text{Fronteiras: } F_{adv}^{bnd} &= \int_{\partial \Omega} \left( \chi_{in} T_{inlet} + (1-\chi_{in}) T \right) \theta (\mathbf{v} \cdot \mathbf{n}) ds
\end{align}
$$

onde $\chi_{in}$ é a função indicadora da fronteira de entrada. *Nota: A integral volumétrica de advecção $-\int_K T \mathbf{v} \cdot \nabla \theta \, dx$ é identicamente zero no código porque $\nabla \theta = \mathbf{0}$ no espaço $DG0$.*

#### Termos de Difusão (SIPG)
$$
\begin{align}
    \text{Interfaces: } F_{dif}^{int} &= \int_{\mathcal{E}_i} \left( - \{D \nabla T\} \cdot \llbracket \theta \rrbracket \mathbf{n} - \{D \nabla \theta\} \cdot \llbracket T \rrbracket \mathbf{n} + \frac{\sigma}{h} \llbracket T \rrbracket \llbracket \theta \rrbracket \right) dS \\
    \text{Fronteiras: } F_{dif}^{bnd} &= \int_{\Gamma_{in}} \left( - (D \nabla T \cdot \mathbf{n})\theta - (D \nabla \theta \cdot \mathbf{n})(T - T_{inlet}) + \frac{\sigma}{h} (T - T_{inlet}) \theta \right) ds
\end{align}
$$

onde $\sigma$ é o parâmetro de penalização e $h$ a dimensão característica da face.

### Discretização e Particularização para DG0
No espaço $DG0$, as funções base são constantes dentro de cada elemento $K$. Portanto:

$$
\nabla T = \mathbf{0} \quad \text{e} \quad \nabla \theta = \mathbf{0} \quad \text{no interior de qualquer } K \in \mathcal{T}_h
$$

Isso simplifica dramaticamente o sistema. A formulação variacional discreta reduz-se apenas à avaliação de fluxos através das faces (as contribuições volumétricas tornam-se nulas, exceto pelo termo fonte, que é 0 neste caso). As formas bilineares $a(T,\theta)$ e lineares $L(\theta)$ extraídas do `Residual` ficam:

$$
\begin{align}
    a(T, \theta) &= \sum_{e \in \mathcal{E}_i} \int_e \left[ \{T\} \llbracket \theta \mathbf{v} \cdot \mathbf{n} \rrbracket + \frac{1}{2} |\mathbf{v} \cdot \mathbf{n}| \llbracket T \rrbracket \llbracket \theta \rrbracket + \frac{\sigma}{h} \llbracket T \rrbracket \llbracket \theta \rrbracket \right] dS \nonumber \\
    &\quad + \sum_{e \in \mathcal{E}_b} \int_e \left[ (1-\chi_{in}) T \theta (\mathbf{v} \cdot \mathbf{n}) + \chi_{in} \frac{\sigma}{h} T \theta \right] ds \\
    L(\theta) &= \int_{\Omega} f \theta dx - \sum_{e \in \mathcal{E}_b} \int_e \chi_{in} T_{inlet} \theta (\mathbf{v} \cdot \mathbf{n}) ds + \sum_{e \in \mathcal{E}_b} \int_e \chi_{in} \frac{\sigma}{h} T_{inlet} \theta ds
\end{align}
$$

### Montagem do Sistema de Equações
A discretização leva ao sistema linear global matricial:

$$
\mathbf{A} \mathbf{T} = \mathbf{b}
$$

#### Montagem Local (Faces)
Como o espaço é $DG0$, cada elemento $K$ possui exatamente $1$ grau de liberdade (sua média). O acoplamento entre incógnitas ocorre exclusivamente através das faces $\mathcal{E}_i$.
Para uma face interna $e$ que conecta o elemento esquerdo $K^+$ e o direito $K^-$, a integral de contorno gera uma matriz de rigidez local do tipo $2 \times 2$:

$$
\mathbf{A}^e_{local} = 
\begin{bmatrix}
    A_{++} & A_{+-} \\
    A_{-+} & A_{--}
\end{bmatrix}
$$

Onde as entradas são calculadas substituindo $\llbracket \theta \rrbracket = \theta^+ \mathbf{n}^+ + \theta^- \mathbf{n}^-$. Avaliando para a advecção upwind (supondo fluxo de $K^+$ para $K^-$, ou seja, $v_n^+ > 0$) e difusão:

$$
\begin{align}
    A_{++} &= \int_e \left( v_n^+ + \frac{\sigma}{h} \right) dS \quad ; \quad A_{+-} = \int_e \left( -\frac{\sigma}{h} \right) dS \\
    A_{-+} &= \int_e \left( -v_n^+ - \frac{\sigma}{h} \right) dS \quad ; \quad A_{--} = \int_e \left( \frac{\sigma}{h} \right) dS
\end{align}
$$

#### Montagem Global
O operador de montagem (assembly) mapeia as matrizes locais das faces (e contornos) para a matriz global esparsa $\mathbf{A} \in \mathbb{R}^{N_e \times N_e}$, onde $N_e$ é o número total de elementos da malha:

$$
\mathbf{A} = \mathbf{A}^{vol} + \sum_{e \in \mathcal{E}_i} \mathcal{A}_i (\mathbf{A}^e_{local}) + \sum_{e \in \mathcal{E}_b} \mathcal{A}_b (\mathbf{A}^{e, bnd}_{local})
$$

Sendo $\mathbf{A}^{vol}$ a matriz nula no caso de advecção-difusão em DG0 sem reação. O sistema final refletirá matrizes esparsas cujos vizinhos correspondem estritamente aos elementos adjacentes compartilhando faces.